# GEPA Prompt Optimization

This notebook optimizes the base agent's system prompt using GEPA, scored by the aligned judge.

GEPA runs one medium optimization pass on a 6-question subset drawn from a
12-question deterministic pool, with a budget of 16 scorer calls. It optimizes only the system prompt in this profile to keep runtime bounded in Free Edition. Results are checkpointed
to a Delta table after every run for resilience against rate limits and interruptions.

The best prompt is registered to the Prompt Registry and promoted to `@production`. Downstream
notebooks consume this prompt:
- `07-AgentSkillsGeneration.ipynb` generates skills using the optimized prompt
- `08_create_agent_with_skills.ipynb` builds the skills-enhanced agent
- `09-Evaluation.ipynb` evaluates both agents on a held-out dataset

**Prerequisites:**
- `03_create_agent_definition.ipynb` has written `agent.py`
- `04-Evaluation.ipynb` has created the evaluation dataset
- `05-JudgeAlignment.ipynb` has produced the aligned judge

In [ ]:
# Avoid reinstalling the full dependency set on every prompt optimization job run.
# If this environment is fresh, install only after import checks fail.
import importlib
import subprocess
import sys

_required_packages = {
    "litellm": "litellm>=1.80.0",
    "databricks_mcp": "databricks-mcp",
    "langgraph": "langgraph",
    "langgraph.checkpoint.postgres": "langgraph-checkpoint-postgres",
    "psycopg": "psycopg[binary,pool]",
    "databricks_langchain": "databricks-langchain",
    "json_repair": "json-repair",
    "gepa": "gepa==0.1.1",
}

def _can_import(module_name):
    try:
        importlib.import_module(module_name)
        return True
    except Exception:
        return False

_missing = [pip_name for import_name, pip_name in _required_packages.items() if not _can_import(import_name)]
if _missing:
    print(f"Installing missing packages: {_missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "--prefer-binary", "-q", *_missing])
    dbutils.library.restartPython()
else:
    print("All prompt optimization dependencies are already available; skipping pip install.")


In [ ]:
import json
import os
import random
import re
import importlib
import hashlib
import warnings
import logging
import time
from contextlib import contextmanager
from pathlib import Path

import litellm
import mlflow

OPENAI_SECRET_SCOPE = os.getenv("OPENAI_SECRET_SCOPE", "<your-secret-scope>")
OPENAI_API_KEY = dbutils.secrets.get(scope=OPENAI_SECRET_SCOPE, key="openai_api_key")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
DIRECT_OPENAI_MODEL_MAP = {
    "databricks/gpt-5-4": "openai/gpt-5.4",
    "databricks/gpt-5-4-nano": "openai/gpt-5.4-nano",
}

def _rewrite_litellm_model(kwargs):
    model = kwargs.get("model")
    if model in DIRECT_OPENAI_MODEL_MAP:
        rewritten = DIRECT_OPENAI_MODEL_MAP[model]
        print(f"Rewriting LiteLLM model {model} -> {rewritten}")
        kwargs["model"] = rewritten

_original_litellm_completion = litellm.completion

def _completion_with_databricks_endpoint_retry(*args, **kwargs):
    _rewrite_litellm_model(kwargs)
    max_attempts = 3
    for attempt in range(max_attempts):
        try:
            return _original_litellm_completion(*args, **kwargs)
        except Exception as exc:
            message = str(exc)
            is_transient_databricks_404 = (
                "ENDPOINT_NOT_FOUND" in message
                and "DatabricksException" in message
            )
            if not is_transient_databricks_404 or attempt == max_attempts - 1:
                raise
            sleep_seconds = 5 * (attempt + 1)
            print(f"Retrying transient Databricks LiteLLM endpoint lookup failure in {sleep_seconds}s ({attempt + 1}/{max_attempts})")
            time.sleep(sleep_seconds)

litellm.completion = _completion_with_databricks_endpoint_retry
print("Installed LiteLLM retry shim for transient Databricks ENDPOINT_NOT_FOUND responses.")

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
SUFFICIENCY_PROMPT_NAME = f"{PROMPT_NAME}_sufficiency"
SYNTHESIS_PROMPT_NAME = f"{PROMPT_NAME}_synthesis"
PARTIAL_SYNTHESIS_PROMPT_NAME = f"{PROMPT_NAME}_partial_synthesis"
# Keep GEPA bounded in Free Edition: optimize the system prompt first.
# Sufficiency and synthesis prompts remain registry-backed and can be optimized separately.
OPTIMIZED_PROMPT_NAMES = [PROMPT_NAME]
REFLECTION_MODEL = CONFIG.get("optimization", {}).get(
    "reflection_model",
    CONFIG["prompt_registry"]["reflection_model"],
)
if REFLECTION_MODEL == "databricks:/gpt-5-4-nano":
    REFLECTION_MODEL = "openai:/gpt-5.4-nano"
OPTIMIZATION_GENERATION_MODEL = CONFIG.get("optimization", {}).get(
    "generation_model",
    REFLECTION_MODEL,
)
JUDGE_MODEL = CONFIG["llm"]["judge_model"]
if JUDGE_MODEL == "databricks:/gpt-5-4":
    JUDGE_MODEL = "openai:/gpt-5.4"
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]

N_RUNS = 1
POOL_SIZE = 20
SUBSET_SIZE = 10
MAX_METRIC_CALLS = 40
USE_LLM_POOL_GENERATION = False
GEPA_RUN_PROFILE = "medium_openai_semantic_v5_sql_tools_system_only"
RANDOM_SEED = 42
FORCE_CURATED_POOL = True

JUDGE_EXPERIMENT_ID = EXPERIMENT_ID

_scope = CONFIG["prompt_registry_auth"]["secret_scope_name"]
_sp_id = dbutils.secrets.get(scope=_scope, key=CONFIG["prompt_registry_auth"]["oauth_client_id_key"])
_sp_secret = dbutils.secrets.get(scope=_scope, key=CONFIG["prompt_registry_auth"]["oauth_client_secret_key"])

@contextmanager
def _sp_auth():
    """Temporarily swap to Service Principal OAuth for prompt registry operations."""
    saved_token = os.environ.pop("DATABRICKS_TOKEN", None)
    os.environ["DATABRICKS_CLIENT_ID"] = _sp_id
    os.environ["DATABRICKS_CLIENT_SECRET"] = _sp_secret
    try:
        yield
    finally:
        os.environ.pop("DATABRICKS_CLIENT_ID", None)
        os.environ.pop("DATABRICKS_CLIENT_SECRET", None)
        if saved_token is not None:
            os.environ["DATABRICKS_TOKEN"] = saved_token

print("SP auth context manager ready.")

with _sp_auth():
    parent_experiment = mlflow.get_experiment(EXPERIMENT_ID)
    experiment_06 = mlflow.set_experiment(f"{parent_experiment.name}-06-prompt-optimization")
EXPERIMENT_ID = experiment_06.experiment_id
print(f"Using dedicated experiment: {experiment_06.name} (ID: {EXPERIMENT_ID})")
print(f"Aligned judge will be loaded from original experiment: {JUDGE_EXPERIMENT_ID}")
print(f"GEPA reflection model: {REFLECTION_MODEL}")

print(f"Experiment config: {N_RUNS} GEPA runs, {SUBSET_SIZE} examples per run, {MAX_METRIC_CALLS} scorer calls per run")
print(f"Total scorer calls budget: ~{N_RUNS * MAX_METRIC_CALLS}")
print(f"Optimizing prompt targets: {OPTIMIZED_PROMPT_NAMES}")
print("Best prompts will be registered to @production for downstream notebooks (07, 08, 09).")

## Step 1: Load or Generate Question Pool

If the pool has been generated before and saved to a Delta table, load it. Otherwise,
use a deterministic fallback pool by default to avoid extra model calls in Free Edition.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

POOL_TABLE = f"{CONFIG['workspace']['catalog']}.{CONFIG['workspace']['schema']}.gepa_optimization_pool_{GEPA_RUN_PROFILE}"

_pool_loaded = False
pool = []
raw_output = "[]"

if not FORCE_CURATED_POOL and spark.catalog.tableExists(POOL_TABLE):
    _pool_df = spark.table(POOL_TABLE)
    _pool_count = _pool_df.count()
    if _pool_count >= POOL_SIZE:
        print(f"Loading existing pool from {POOL_TABLE} ({_pool_count} rows)")
        pool = [json.loads(row["example_json"]) for row in _pool_df.orderBy("idx").collect()]
        pool = pool[:POOL_SIZE]
        _pool_loaded = True
    else:
        print(f"Pool table exists but only has {_pool_count} rows (need {POOL_SIZE}). Regenerating.")

if not _pool_loaded and USE_LLM_POOL_GENERATION:
    from openai import OpenAI
    from databricks.sdk import WorkspaceClient

    w = WorkspaceClient()
    fmapi_client: OpenAI = w.serving_endpoints.get_open_ai_client()

    OPTIMIZATION_GEN_MODEL = OPTIMIZATION_GENERATION_MODEL.replace("databricks:/", "")
    print(f"Generating {POOL_SIZE}-question pool with model: {OPTIMIZATION_GEN_MODEL}")

    GENERATION_PROMPT = f"""You are an expert at generating realistic user questions for a baseball hitting analysis AI assistant.

The assistant helps batters prepare for matchups against specific pitchers. It can look up player IDs, get pitcher tendencies by count/hand/runners, pull historical batter-pitcher matchups, look up arsenals, recommend lineups, and query a Genie Space for SQL-based analytics.

Generate exactly {POOL_SIZE} training examples as a JSON array. Each example should be an object with exactly two keys:
- "inputs": a realistic user question (single string)
- "expected_response": a 1-2 sentence description of what a good response should contain

Write questions the way a real hitting coach, analyst, or front office person would ask them. Use answerable requests that map to known UC functions or Genie table analysis. Avoid player similarity, vector search, team-wide advanced aggregate questions, and rare metrics not represented in the tool outputs. Cover:
- Quick factual lookups ("What does Gerrit Cole throw?")
- Matchup scouting ("How will Max Fried attack George Springer?")
- Situational strategy ("How does Corbin Burnes change with runners in scoring position?")
- Lineup construction ("Who should I start against Logan Webb?")
- Open-ended at-bat previews ("What should Freddie Freeman expect facing Zac Gallen?")
- Pitch tendency and Statcast-style questions when they can be answered from pitch type, count, location, release speed, or spin rate evidence

Use real MLB player names and teams. Use 3-letter abbreviations: TEX,CHC,LAA,LAD,STL,PHI,ARI,OAK,TBR,MIN,CLE,CHW,NYM,COL,SEA,MIA,SDP,WSN,HOU,SFG,CIN,BAL,KCR,PIT,ATL,NYY,DET,MIL,TOR,BOS,ATH

Only reference 2024 & 2025 seasons. Keep "expected_response" to 1-2 sentences.
Ensure all {POOL_SIZE} questions are distinct and cover a broad range of the scenarios above.

Return ONLY a valid JSON array, no other text."""

    response = fmapi_client.chat.completions.create(
        model=OPTIMIZATION_GEN_MODEL,
        messages=[{"role": "user", "content": GENERATION_PROMPT}],
        temperature=0.9,
        max_tokens=30000,
    )

    raw_output = response.choices[0].message.content
    print(f"Raw output length: {len(raw_output)} chars")
    print(f"First 500 chars:\n{raw_output[:500]}")
elif not _pool_loaded:
    print("Skipping LLM-generated optimization pool; using deterministic fallback examples.")

In [ ]:
if not _pool_loaded:
    from json_repair import repair_json

    def _extract_json_array(text: str):
        cleaned = re.sub(r'^```(?:json)?\s*', '', text.strip())
        cleaned = re.sub(r'\s*```$', '', cleaned)

        decoder = json.JSONDecoder()
        candidates = []
        for match in re.finditer(r'\[', cleaned):
            try:
                obj, end = decoder.raw_decode(cleaned[match.start():])
                if isinstance(obj, list):
                    candidates.append(obj)
            except json.JSONDecodeError:
                continue
        if candidates:
            return max(candidates, key=len)

        json_match = re.search(r'\[.*\]', cleaned, re.DOTALL)
        candidate = json_match.group() if json_match else cleaned
        repaired = repair_json(candidate, return_objects=True)
        if isinstance(repaired, dict) and "examples" in repaired:
            repaired = repaired["examples"]
        if not isinstance(repaired, list):
            raise ValueError(f"Expected JSON list, got {type(repaired).__name__}")
        return repaired

    def _fallback_examples():
        return [
            {
                "inputs": "What are Corbin Burnes' pitch tendencies with a runner on second against left-handed batters in 2025?",
                "expected_response": "A good response should use runner-state tendency evidence, mention pitch mix and locations, flag sample size if small, and give an actionable left-handed hitter plan.",
            },
            {
                "inputs": "What are Max Fried's tendencies in 1-1 counts against right-handed batters in 2025?",
                "expected_response": "A good response should summarize the retrieved 1-1 count pitch mix and locations against right-handed batters with raw percentages and a hitter takeaway.",
            },
            {
                "inputs": "How will Spencer Strider pitch to Juan Soto in 2024?",
                "expected_response": "A good response should combine matchup and pitcher tendency evidence into a practical plan for Soto without claiming unavailable data.",
            },
            {
                "inputs": "How will Spencer Strider pitch to Juan Soto in 2024, and what was Strider's spin rate on his fastball in 2024?",
                "expected_response": "A good response should answer the matchup portion and separately answer or qualify the fastball spin-rate portion using available evidence.",
            },
            {
                "inputs": "What should Freddie Freeman expect facing Zac Gallen in 2024?",
                "expected_response": "A good response should summarize matchup history, likely pitch types and locations, and give Freeman an approach recommendation.",
            },
            {
                "inputs": "What pitch types does Logan Webb throw in 2025, and what should hitters expect?",
                "expected_response": "A good response should describe Webb's pitch repertoire, relative usage, and practical implications for hitters.",
            },
            {
                "inputs": "How does Gerrit Cole pitch with runners on base against right-handed batters in 2025?",
                "expected_response": "A good response should use runner-state tendency evidence and give right-handed hitters a plan for likely pitches and zones.",
            },
            {
                "inputs": "What are Zack Wheeler's tendencies in 3-2 counts against left-handed batters in 2025?",
                "expected_response": "A good response should summarize 3-2 pitch mix and location tendencies against left-handed batters and give a two-strike approach.",
            },
            {
                "inputs": "For Aaron Judge against Framber Valdez in 2025, what pitch mix and locations should Judge prepare for?",
                "expected_response": "A good response should use available matchup and pitcher tendency data to describe expected pitch mix, locations, and Judge's plan.",
            },
            {
                "inputs": "What are Yu Darvish's main pitch types in 2025, and where does he tend to locate them?",
                "expected_response": "A good response should summarize Darvish's pitch repertoire and location tendencies with actionable implications.",
            },
            {
                "inputs": "How does Blake Snell usually attack Mookie Betts in 2024?",
                "expected_response": "A good response should answer from available matchup or tendency data and clearly state any sample-size limits.",
            },
            {
                "inputs": "What are Aaron Nola's pitch tendencies against left-handed batters in 2025?",
                "expected_response": "A good response should summarize Nola's pitch mix and zones against left-handed batters and recommend a hitter approach.",
            },
        ][:POOL_SIZE]


    try:
        generated_examples = _extract_json_array(raw_output)
        print(f"Parsed {len(generated_examples)} examples from FMAPI")
    except Exception as parse_error:
        print(f"WARNING: Could not parse generated optimization pool: {parse_error}")
        print("Using deterministic fallback optimization pool.")
        generated_examples = _fallback_examples()

    valid_examples = []
    for ex in generated_examples:
        if not isinstance(ex, dict):
            continue
        question = ex.get("inputs") or ex.get("input") or ex.get("question")
        if isinstance(question, dict):
            input_list = question.get("input", [])
            question = next((m.get("content") for m in input_list if isinstance(m, dict) and m.get("role") == "user"), "")
        if not question:
            continue
        raw_expectation = ex.get("expected_response") or ex.get("expectations") or ex.get("expected") or "A good response should answer with grounded baseball analysis and an actionable recommendation."
        valid_examples.append({"inputs": str(question), "expected_response": raw_expectation})

    if len(valid_examples) < POOL_SIZE:
        print(f"WARNING: Got {len(valid_examples)} valid generated examples instead of {POOL_SIZE}. Filling from fallback pool.")
        seen = {ex["inputs"] for ex in valid_examples}
        for ex in _fallback_examples():
            if ex["inputs"] not in seen:
                valid_examples.append(ex)
                seen.add(ex["inputs"])
            if len(valid_examples) >= POOL_SIZE:
                break

    generated_examples = valid_examples[:POOL_SIZE]
    actual_subset_size = min(SUBSET_SIZE, len(generated_examples))

    for ex in generated_examples:
        raw_expectation = ex.get("expected_response") or ex.get("expectations", "")
        if isinstance(raw_expectation, dict):
            expectations_dict = raw_expectation
        else:
            expectations_dict = {"expected_response": str(raw_expectation)}

        pool.append({
            "inputs": {
                "input": [{"role": "user", "content": ex["inputs"]}]
            },
            "expectations": expectations_dict,
        })

    random.seed(RANDOM_SEED)
    random.shuffle(pool)

    _pool_schema = StructType([
        StructField("idx", IntegerType()),
        StructField("example_json", StringType()),
    ])
    _pool_rows = [{"idx": i, "example_json": json.dumps(p)} for i, p in enumerate(pool)]
    spark.createDataFrame(_pool_rows, schema=_pool_schema).write.mode("overwrite").saveAsTable(POOL_TABLE)
    print(f"Saved {len(pool)} examples to {POOL_TABLE}")
else:
    actual_subset_size = SUBSET_SIZE

subsets = [pool[i * actual_subset_size:(i + 1) * actual_subset_size] for i in range(N_RUNS)]
subsets = [s for s in subsets if s]

print(f"\nPool: {len(pool)} examples, {len(subsets)} disjoint subsets of {actual_subset_size} each")
for i, s in enumerate(subsets):
    q = s[0]["inputs"]["input"][0]["content"]
    print(f"  Subset {i}: '{q[:80]}...'")

assert pool and isinstance(pool[0]["expectations"], dict), "expectations must be a dict!"
print("\nStructure validation passed")


## Step 2: Load Aligned Judge

In [ ]:
from mlflow.genai.scorers import Scorer
from mlflow.genai.judges import make_judge
from mlflow.entities import Feedback
import re as _re
import sys as _sys
import litellm as _litellm

GUARDRAIL_SENTINEL = -1.0
_GUARDRAIL_RE = _re.compile(r'(?<![_a-zA-Z])arsenal(?![_a-zA-Z])', _re.IGNORECASE)


def _scrub(text):
    """Replace guardrail-triggering baseball terms in a string."""
    if isinstance(text, str):
        return _GUARDRAIL_RE.sub('pitch repertoire', text)
    return text


def _scrub_messages(messages):
    if not messages or not isinstance(messages, list):
        return
    for msg in messages:
        if not isinstance(msg, dict):
            continue
        c = msg.get("content")
        if isinstance(c, str):
            msg["content"] = _scrub(c)
        elif isinstance(c, list):
            for part in c:
                if isinstance(part, dict) and isinstance(part.get("text"), str):
                    part["text"] = _scrub(part["text"])


_original_completion = _litellm.completion

def _sanitized_completion(*args, **kwargs):
    _rewrite_litellm_model(kwargs)
    _scrub_messages(kwargs.get("messages"))
    if args:
        for a in args:
            if isinstance(a, list):
                _scrub_messages(a)
    return _original_completion(*args, **kwargs)

_litellm.completion = _sanitized_completion

_patched_count = 0
for _mod_name in list(_sys.modules.keys()):
    _mod = _sys.modules.get(_mod_name)
    if _mod is None:
        continue
    for _attr in ['completion', 'litellm_completion']:
        try:
            _ref = getattr(_mod, _attr, None)
            if _ref is _original_completion:
                setattr(_mod, _attr, _sanitized_completion)
                _patched_count += 1
        except Exception:
            pass

print(f"Patched litellm.completion + {_patched_count} cached refs across loaded modules")

ALIGNED_SEMANTIC_GUIDELINES = """
Aligned judge guidelines distilled from the reviewed traces:
1. Define or enumerate pitch categories when the question asks for grouped pitch-type percentages, such as breaking balls.
2. Assume standard Statcast and Genie fields are available when tools or traces show evidence, including pitch distribution, release speed, spin rate, and situational splits.
3. Never identify players only by internal IDs; player names must be resolved and used in the final answer.
4. If tools partially fail or time out, still use available evidence for a best-effort baseball takeaway rather than claiming no data.
5. Raw retrieved evidence is not enough; the answer must synthesize tendencies and give an actionable hitter or coaching recommendation.
6. Respect concise requests, but concise answers can still be complete.
7. Check numerical consistency and pitch labels or abbreviations.
8. For repertoire and tendency questions, usage distribution is usually expected unless the user explicitly asks for a very concise answer.
"""

_inner_judge = make_judge(
    name=ALIGNED_JUDGE_NAME,
    instructions=(
        "Evaluate if the response in {{ outputs }} appropriately analyzes the available data and provides an actionable recommendation "
        "to the question in {{ inputs }}. The response should be accurate, contextually relevant, and give a strategic advantage to the "
        "hitter or coaching staff making the request. "
        "Your grading criteria should be: "
        "1: Completely unacceptable. Incorrect data interpretation or no recommendations. "
        "2: Mostly unacceptable. Irrelevant or spurious feedback or weak recommendations provided with minimal strategic advantage. "
        "3: Somewhat acceptable. Relevant feedback provided with some strategic advantage. "
        "4: Mostly acceptable. Relevant feedback provided with strong strategic advantage. "
        "5: Completely acceptable. Relevant feedback provided with excellent strategic advantage.\n\n"
        + ALIGNED_SEMANTIC_GUIDELINES
    ),
    feedback_value_type=float,
    model=JUDGE_MODEL,
)
print(f"Created inline aligned semantic judge: {_inner_judge.name} using {JUDGE_MODEL}")

_patched_count2 = 0
for _mod_name in list(_sys.modules.keys()):
    _mod = _sys.modules.get(_mod_name)
    if _mod is None:
        continue
    for _attr in ['completion', 'litellm_completion']:
        try:
            _ref = getattr(_mod, _attr, None)
            if _ref is _original_completion:
                setattr(_mod, _attr, _sanitized_completion)
                _patched_count2 += 1
        except Exception:
            pass
if _patched_count2:
    print(f"Patched {_patched_count2} additional refs found after scorer load")


class GuardrailSafeScorer(Scorer):
    """Wraps an existing scorer and catches AI Gateway guardrail errors.

    Returns Feedback with value=GUARDRAIL_SENTINEL (-1.0) so the objective
    function can detect and exclude these from the running average.
    Also scrubs inputs/outputs before delegating and detects predict_fn fallbacks.
    """
    name: str = ALIGNED_JUDGE_NAME
    _delegate: object = None

    class Config:
        underscore_attrs_are_private = True

    def __init__(self, delegate, **kwargs):
        super().__init__(**kwargs)
        self._delegate = delegate

    def __call__(self, *, inputs=None, outputs=None, expectations=None, trace=None):
        if isinstance(outputs, str) and "(Skipped: input guardrail triggered" in outputs:
            logging.warning("predict_fn returned guardrail fallback, marking as sentinel")
            return Feedback(
                name=self.name,
                value=GUARDRAIL_SENTINEL,
                rationale="GUARDRAIL_SKIP",
            )
        scrubbed_outputs = _scrub(outputs) if isinstance(outputs, str) else outputs
        try:
            return self._delegate(
                inputs=inputs, outputs=scrubbed_outputs,
                expectations=expectations, trace=trace,
            )
        except Exception as e:
            if "guardrail" in str(e).lower() or "input_guardrail_triggered" in str(e):
                logging.warning(f"Guardrail triggered during scoring: {str(e)[:120]}")
                return Feedback(
                    name=self.name,
                    value=GUARDRAIL_SENTINEL,
                    rationale="GUARDRAIL_SKIP",
                )
            raise



class DataToolAvailabilityScorer(Scorer):
    """Scores retrieval/tool availability separately from answer quality."""
    name: str = "data_tool_availability"

    def __call__(self, *, inputs=None, outputs=None, expectations=None, trace=None):
        text = json.dumps(outputs, default=str).lower()
        timeout_patterns = [
            "execution timed out",
            "timed out after",
            "read timed out",
            "provisioning resources",
            "daily limit",
            "endpoint_not_found",
            "sql_execution_exception",
        ]
        usable_patterns = [
            "[tool outputs]",
            "function_call_output",
            '"rows":[[',
            "pitch_count",
            "frequency_pct",
            "avg_release_speed",
            "avg_release_spin_rate",
        ]
        no_data_patterns = [
            "no relevant records",
            "no rows",
            "empty result",
            "don't have any information",
            "does not include information",
        ]
        if any(p in text for p in timeout_patterns):
            return Feedback(name=self.name, value=0.0, rationale="Execution failure or timeout blocked reliable data access.")
        if any(p in text for p in usable_patterns):
            return Feedback(name=self.name, value=1.0, rationale="Usable retrieved evidence is present.")
        if any(p in text for p in no_data_patterns):
            return Feedback(name=self.name, value=0.25, rationale="Run completed but available data appears limited or empty.")
        return Feedback(name=self.name, value=0.5, rationale="Data availability is unclear.")


data_tool_availability_scorer = DataToolAvailabilityScorer()
print("Created data/tool availability scorer")


aligned_judge = GuardrailSafeScorer(delegate=_inner_judge)
print(f"Wrapped judge with guardrail-safe handler (sentinel={GUARDRAIL_SENTINEL})")

## Step 3: Define Shared Helpers

The `predict_fn_factory` creates a predict function bound to a specific agent module,
letting us switch between `agent` (base) and `agent_with_skills` without code duplication.

In [ ]:
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic')
logging.getLogger('mlflow.genai.judges.instructions_judge').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.fluent').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.export.mlflow_v3').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.provider').setLevel(logging.ERROR)


def _get(item, key, default=""):
    if isinstance(item, dict):
        return item.get(key, default)
    return getattr(item, key, default)


def _compact_json(value, limit=700):
    text = value if isinstance(value, str) else json.dumps(value, default=str)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit] + "..." if len(text) > limit else text


def _extract_compact_response(result) -> str:
    tool_calls = []
    tool_outputs = []
    final_text = ""

    for item in result.output:
        item_type = _get(item, "type")

        if item_type == "function_call":
            name = _get(item, "name", "unknown")
            args_str = _get(item, "arguments", "{}")
            tool_calls.append(f"  - {name}({_compact_json(args_str, 300)})")

        elif item_type == "function_call_output":
            call_id = _get(item, "call_id", "unknown")
            output = _get(item, "output", "")
            status = "error" if "error" in str(output).lower() or "timed out" in str(output).lower() else "ok"
            tool_outputs.append(f"  - call_id={call_id} status={status} output={_compact_json(output, 700)}")

        elif item_type == "message":
            content = _get(item, "content", [])
            if isinstance(content, list):
                for block in content:
                    block_type = _get(block, "type") if isinstance(block, dict) else getattr(block, "type", "")
                    if block_type == "output_text":
                        text = _get(block, "text", "") if isinstance(block, dict) else getattr(block, "text", "")
                        if text:
                            final_text = text
            elif isinstance(content, str):
                final_text = content

        elif item_type == "text":
            text = _get(item, "text", "")
            if text:
                final_text = text

    parts = []
    if tool_calls:
        parts.append("[Tool Calls]\n" + "\n".join(tool_calls))
    if tool_outputs:
        parts.append("[Tool Outputs]\n" + "\n".join(tool_outputs))
    if final_text:
        parts.append("[Agent Analysis]\n" + final_text)

    compact = "\n\n".join(parts) if parts else "(no response)"
    return _scrub(compact)


def _feedback_value(scores, name):
    feedback = scores.get(name)
    if feedback and hasattr(feedback, 'feedback') and hasattr(feedback.feedback, 'value'):
        try:
            return float(feedback.feedback.value)
        except (ValueError, TypeError):
            return None
    return None


def make_objective_function(judge_name, subset_size):
    """Create an objective function scoped to a specific run's subset size.

    Guardrail rows are skipped. Rows with unavailable data/tool execution are
    neutralized so prompt search is not rewarded or penalized for infrastructure
    or empty-table conditions.
    """
    counter = {"count": 0, "scores": [], "skipped": 0, "neutralized": 0, "total_seen": 0}

    def objective_function(scores: dict) -> float:
        feedback = scores.get(judge_name)
        counter["total_seen"] += 1
        raw_score = _feedback_value(scores, judge_name)
        availability = _feedback_value(scores, "data_tool_availability")

        is_sentinel = raw_score is not None and raw_score == GUARDRAIL_SENTINEL
        is_skip = (
            is_sentinel
            or (feedback and hasattr(feedback, 'feedback')
                and getattr(feedback.feedback, 'rationale', '') == "GUARDRAIL_SKIP")
        )

        if is_skip:
            counter["skipped"] += 1
            if counter["scores"]:
                return sum(counter["scores"]) / len(counter["scores"])
            return float("nan")

        if availability is not None and availability < 0.5:
            counter["neutralized"] += 1
            neutral_score = sum(counter["scores"]) / len(counter["scores"]) if counter["scores"] else 0.5
            if counter["total_seen"] == subset_size:
                counter["count"] += 1
                avg = sum(counter["scores"]) / len(counter["scores"]) if counter["scores"] else neutral_score
                print(f"    Candidate #{counter['count']} avg: {avg:.4f} (availability-neutralized: {counter['neutralized']}, guardrail-skipped: {counter['skipped']})")
                counter["scores"] = []
                counter["skipped"] = 0
                counter["neutralized"] = 0
                counter["total_seen"] = 0
            return neutral_score

        if raw_score is not None:
            normalized = raw_score / 5.0
            counter["scores"].append(normalized)

            if counter["total_seen"] == subset_size:
                n_real = len(counter["scores"])
                avg = sum(counter["scores"]) / n_real if n_real else 0.0
                counter["count"] += 1
                print(f"    Candidate #{counter['count']} avg: {avg:.4f} (0-1 scale) "
                      f"({n_real} scored, {counter['neutralized']} availability-neutralized, {counter['skipped']} guardrail-skipped)")
                counter["scores"] = []
                counter["skipped"] = 0
                counter["neutralized"] = 0
                counter["total_seen"] = 0

            return normalized

        return float("nan")

    return objective_function


def _format_prompt_obj(prompt_obj):
    return prompt_obj.format() if hasattr(prompt_obj, "format") else str(prompt_obj)


def predict_fn_factory(agent_module_name: str, prompt_refs=None):
    """Return a predict_fn bound to the given agent module and prompt refs.

    GEPA can intercept every mlflow.genai.load_prompt call below because each
    URI is also passed through prompt_uris in optimize_prompts.
    """
    mod = importlib.import_module(agent_module_name)
    agent_instance = mod.AGENT
    last_hash = {"hash": None}
    _prompt_refs = prompt_refs or seed_prompt_refs
    _prompt_uris = {name: ref.uri for name, ref in _prompt_refs.items()}

    def predict_fn(input):
        prompts = {name: mlflow.genai.load_prompt(uri) for name, uri in _prompt_uris.items()}
        system_content = _format_prompt_obj(prompts[PROMPT_NAME])

        prompt_hash = hashlib.md5("\n".join(_format_prompt_obj(p) for p in prompts.values()).encode()).hexdigest()
        if prompt_hash != last_hash["hash"]:
            print(f"    New prompt candidate set (hash: {prompt_hash[:8]})")
            last_hash["hash"] = prompt_hash

        agent_instance.system_prompt = prompts[PROMPT_NAME]
        mod.SYSTEM_PROMPT = prompts[PROMPT_NAME]
        if SUFFICIENCY_PROMPT_NAME in prompts:
            mod.SUFFICIENCY_PROMPT = prompts[SUFFICIENCY_PROMPT_NAME]
        if SYNTHESIS_PROMPT_NAME in prompts:
            mod.SYNTHESIS_PROMPT = prompts[SYNTHESIS_PROMPT_NAME]
        if PARTIAL_SYNTHESIS_PROMPT_NAME in prompts:
            mod.PARTIAL_SYNTHESIS_PROMPT = prompts[PARTIAL_SYNTHESIS_PROMPT_NAME]

        if isinstance(input, dict) and "input" in input:
            user_message = input["input"][0]["content"]
        elif isinstance(input, list):
            user_message = input[0]["content"]
        else:
            user_message = str(input)

        messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_message},
        ]

        mlflow.tracing.disable()
        try:
            result = agent_instance.predict({"input": messages})
        except Exception as e:
            mlflow.tracing.enable()
            err_str = str(e)
            if "input_guardrail_triggered" in err_str or "guardrail" in err_str.lower():
                logging.warning(f"Guardrail false positive, returning fallback: {user_message[:80]}")
                return "[Agent Analysis]\n(Skipped: input guardrail triggered on benign baseball query)"
            raise
        finally:
            mlflow.tracing.enable()

        return _extract_compact_response(result)

    return predict_fn


SEED_PROMPT_URI_OVERRIDES = CONFIG.get("optimization", {}).get("seed_prompt_uris", {})
with _sp_auth():
    seed_prompt_refs = {
        name: mlflow.genai.load_prompt(SEED_PROMPT_URI_OVERRIDES.get(name, f"prompts:/{name}@production"))
        for name in OPTIMIZED_PROMPT_NAMES
    }

print("Seed prompts loaded from @production unless overridden:")
for name, ref in seed_prompt_refs.items():
    print(f"  {name}: {ref.uri}")
print("GEPA optimizes from these prompt refs. Best result is registered to @production.")
print("Helpers defined.")


## Step 4: Sanity Check

Run a single prediction with the base agent to verify extraction works before the full experiment.

In [ ]:
test_input = subsets[0][0]["inputs"]
q_text = test_input["input"][0]["content"]
print(f"Test question: {q_text[:150]}")

print("\n--- Testing agent (base) ---")
pfn = predict_fn_factory("agent")
output = pfn(test_input)
has_tools = "[Tool Calls]" in output
has_analysis = "[Agent Analysis]" in output
print(f"  Output length: {len(output)} chars")
print(f"  Has [Tool Calls]: {has_tools}")
print(f"  Has [Agent Analysis]: {has_analysis}")
if not has_analysis:
    print(f"  WARNING: No [Agent Analysis] found. Check agent output format.")
else:
    print(f"  OK")

## Step 5: Run GEPA Optimization (with checkpoint/resume)

5 GEPA runs on the base agent (`agent.py`), starting from the original prompt (v1).
Each run uses a disjoint 20-question subset and 100 scorer calls.

Results are checkpointed to a Delta table after every run. If the notebook is interrupted
(e.g., rate limits), re-running this cell resumes from where it left off.

In [ ]:
from mlflow.genai.optimize import GepaPromptOptimizer
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import time

CHECKPOINT_TABLE = f"{CONFIG['workspace']['catalog']}.{CONFIG['workspace']['schema']}.gepa_experiment_checkpoint_{GEPA_RUN_PROFILE}"

_checkpoint_schema = StructType([
    StructField("agent_type", StringType()),
    StructField("run_idx", IntegerType()),
    StructField("initial_score", DoubleType()),
    StructField("final_score", DoubleType()),
    StructField("prompt_template", StringType()),
    StructField("prompt_templates_json", StringType()),
    StructField("elapsed_seconds", DoubleType()),
])

if spark.catalog.tableExists(CHECKPOINT_TABLE):
    _checkpoint_df = spark.table(CHECKPOINT_TABLE)
    results = [row.asDict() for row in _checkpoint_df.collect()]
    print(f"Resumed: {len(results)} completed runs loaded from {CHECKPOINT_TABLE}")
else:
    results = []
    print(f"Starting fresh experiment (no table {CHECKPOINT_TABLE})")


def _save_checkpoint():
    df = spark.createDataFrame(results, schema=_checkpoint_schema)
    df.write.mode("overwrite").saveAsTable(CHECKPOINT_TABLE)
    print(f"  Checkpoint saved to {CHECKPOINT_TABLE} ({len(results)} runs)")


def _is_done(agent_type, run_idx):
    return any(r["agent_type"] == agent_type and r["run_idx"] == run_idx for r in results)


def _get_result(agent_type, run_idx):
    for r in results:
        if r["agent_type"] == agent_type and r["run_idx"] == run_idx:
            return r
    return None


def _fmt(score):
    return f"{score:.3f}" if score is not None else "N/A"


experiment_start = time.time()

# Phase A: base agent, seed = @production prompt set
print(f"\n{'=' * 80}")
print(f"PHASE A: BASE AGENT (agent) -- seed: @production prompt set")
print(f"{'=' * 80}")

phase_a_remaining = [i for i in range(N_RUNS) if not _is_done("base", i)]
if phase_a_remaining:
    predict_fn_base = predict_fn_factory("agent")

for run_idx in range(N_RUNS):
    subset = subsets[run_idx]

    if _is_done("base", run_idx):
        r = _get_result("base", run_idx)
        print(f"\n--- [base] Run {run_idx + 1}/{N_RUNS} --- SKIPPED (complete: {_fmt(r['initial_score'])} -> {_fmt(r['final_score'])})")
        continue

    run_start = time.time()
    print(f"\n--- [base] Run {run_idx + 1}/{N_RUNS} ---")
    print(f"  Dataset: subset {run_idx} ({len(subset)} questions)")

    with _sp_auth():
        prompt_refs = {name: mlflow.genai.load_prompt(seed_prompt_refs[name].uri) for name in OPTIMIZED_PROMPT_NAMES}
    obj_fn = make_objective_function(ALIGNED_JUDGE_NAME, len(subset))

    result = mlflow.genai.optimize_prompts(
        predict_fn=predict_fn_base,
        train_data=subset,
        prompt_uris=[prompt_refs[name].uri for name in OPTIMIZED_PROMPT_NAMES],
        optimizer=GepaPromptOptimizer(
            reflection_model=REFLECTION_MODEL,
            max_metric_calls=MAX_METRIC_CALLS,
            display_progress_bar=False,
        ),
        scorers=[aligned_judge],
        aggregation=obj_fn,
    )

    initial = getattr(result, 'initial_eval_score', None)
    final = getattr(result, 'final_eval_score', None)
    elapsed = time.time() - run_start

    results.append({
        "agent_type": "base",
        "run_idx": run_idx,
        "initial_score": initial,
        "final_score": final,
        "prompt_template": result.optimized_prompts[0].template,
        "prompt_templates_json": json.dumps({name: prompt.template for name, prompt in zip(OPTIMIZED_PROMPT_NAMES, result.optimized_prompts)}),
        "elapsed_seconds": elapsed,
    })
    _save_checkpoint()
    print(f"  Result: {_fmt(initial)} -> {_fmt(final)} ({elapsed:.0f}s)")

total_elapsed = time.time() - experiment_start
print(f"\n{'=' * 80}")
print(f"GEPA OPTIMIZATION COMPLETE ({total_elapsed / 60:.1f} minutes)")
print(f"Total runs: {len(results)}")
print(f"{'=' * 80}")

## Step 6: Identify Best Prompt and Summarize

Find the best GEPA run and display results. The best prompt is then registered in Step 7.

In [ ]:
import pandas as pd

phase_a_results = [r for r in results if r["agent_type"] == "base"]
best_run = max(phase_a_results, key=lambda r: r["final_score"] or 0)
best_prompt_templates = json.loads(best_run.get("prompt_templates_json") or json.dumps({PROMPT_NAME: best_run["prompt_template"]}))
best_prompt_text = best_prompt_templates.get(PROMPT_NAME, best_run["prompt_template"])

gepa_df = pd.DataFrame(phase_a_results)
gepa_df["lift"] = gepa_df["final_score"] / gepa_df["initial_score"]

base_init = gepa_df["initial_score"].dropna()
base_final = gepa_df["final_score"].dropna()
base_lifts = gepa_df["lift"].dropna()

print("GEPA OPTIMIZATION RESULTS")
print("=" * 90)
print(gepa_df[["run_idx", "initial_score", "final_score", "lift", "elapsed_seconds"]].to_string(index=False, float_format="%.4f"))

print(f"\n  Initial (1-5):  {base_init.mean()*5:.2f} +/- {base_init.std()*5:.2f}")
print(f"  Final (1-5):    {base_final.mean()*5:.2f} +/- {base_final.std()*5:.2f}")
print(f"  Lift:           {base_lifts.mean():.3f}x +/- {base_lifts.std():.3f}")

print(f"\n  Best run: {best_run['run_idx']} "
      f"({best_run['initial_score']:.3f} -> {best_run['final_score']:.3f})")
print(f"  Prompt hash: {hashlib.md5(best_prompt_text.encode()).hexdigest()[:12]}")

print(f"\nNext: register this prompt (@production), then run 09-Evaluation.ipynb for held-out comparison.")

## Step 7: Register Best Prompt

Register the best prompt and promote to `@production`.

In [ ]:
print(f"Best run: [base] run {best_run['run_idx']}")
print(f"  Score: {best_run['initial_score']:.3f} -> {best_run['final_score']:.3f}")
print(f"  Lift: {best_run['final_score'] / best_run['initial_score']:.2f}x")
print("  Optimized prompts:")
for prompt_name, prompt_text in best_prompt_templates.items():
    print(f"  - {prompt_name}: {len(prompt_text)} chars, hash={hashlib.md5(prompt_text.encode()).hexdigest()[:12]}")

registered_prompts = {}
for prompt_name, prompt_text in best_prompt_templates.items():
    new_prompt = mlflow.genai.register_prompt(
        name=prompt_name,
        template=prompt_text,
        commit_message=(
            f"Best prompt from GEPA experiment "
            f"(run={best_run['run_idx']}, "
            f"score: {best_run['initial_score']:.3f} -> {best_run['final_score']:.3f}, "
            f"judge: {ALIGNED_JUDGE_NAME})"
        ),
        tags={
            "experiment": "gepa_optimization",
            "agent_type": "base",
            "run_idx": str(best_run["run_idx"]),
            "initial_score": str(best_run["initial_score"]),
            "final_score": str(best_run["final_score"]),
            "judge": ALIGNED_JUDGE_NAME,
            "gepa_run_profile": GEPA_RUN_PROFILE,
        },
    )
    mlflow.genai.set_prompt_alias(
        name=prompt_name,
        alias="production",
        version=new_prompt.version,
    )
    registered_prompts[prompt_name] = new_prompt.version
    print(f"Registered {prompt_name} version {new_prompt.version} and promoted to @production")


## Step 8: Save Raw Results

Display the checkpoint table. Run `09-Evaluation.ipynb` for the held-out comparison.

In [ ]:
print(f"GEPA results persisted in Delta table: {CHECKPOINT_TABLE}")
print(f"Total rows: {spark.table(CHECKPOINT_TABLE).count()}")
display(spark.table(CHECKPOINT_TABLE).select("agent_type", "run_idx", "initial_score", "final_score", "elapsed_seconds"))

print(f"\nOptimization pool persisted in: {POOL_TABLE}")
print(f"Prompts registered to @production: {registered_prompts}")
print("Run 09-Evaluation.ipynb for held-out comparison.")